<a href="https://colab.research.google.com/github/Amshu0-0/CA/blob/logbert-only/NLP/notebooks/colab_full_run.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Full experiment on a GPU (Google Colab)

This notebook runs the whole project on the **full BGL dataset**:

1. download and preprocess BGL (Drain templates, chronological 60/20/20 split, 5-minute windows)
2. train LogBERT and tune its detection settings on the validation period
3. compare the two LogBERT variants on the same test windows and run the error analysis
4. launch the demo

**Runtime:** choose *Runtime -> Change runtime type -> T4 GPU*. Expect roughly 1-2 hours in total;
the preprocessing and the training dominate. Copy `outputs/` to Google Drive at the end (last but one cell),
because a Colab session is erased when it ends.

**Before running:** Do not select Run all without reviewing the cells. Training uses GPU resources.
`notebooks/train_on_gpu.ipynb` is the longer notebook of the same run: it also keeps the work in Google Drive
across disconnects and draws the tables and figures.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 0. Get the code

In [ ]:
BRANCH = "logbert-only"
!git clone -q -b {BRANCH} https://github.com/Amshu0-0/CA.git
%cd CA/NLP
!pip install -q -e ".[demo]"

## 1. Data
Downloads BGL (4,747,963 messages) from Loghub's Zenodo record and preprocesses it.

In [ ]:
!python scripts/download_data.py
!python scripts/preprocess.py

In [ ]:
import json

stats = json.load(open("data/processed/bgl/stats.json"))
print(json.dumps(stats["splits"], indent=2))

## 2. LogBERT
Training uses only normal windows from the training period; early stopping watches a held-out tail of
those windows. Detection settings `g` and `r` are then tuned on the labeled validation period, for the
original random masking (A) and for our full-coverage extension (A+).

In [ ]:
!python scripts/train_logbert.py
!python scripts/evaluate_logbert.py
!python scripts/evaluate_logbert.py --set logbert.detection.full_coverage=true

## 3. Comparison and error analysis

In [ ]:
!python scripts/compare.py
!python scripts/analyze.py

In [ ]:
from IPython.display import Markdown, display

display(Markdown(open("outputs/bgl/results/comparison.md").read()))
display(Markdown(open("outputs/bgl/analysis/analysis.md").read()))

## 4. Keep the results
Copies `outputs/` (checkpoint, predictions, metrics) to Google Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/CA-NLP
!cp -r outputs /content/drive/MyDrive/CA-NLP/

## 5. Live demo
Prints a public `gradio.live` link you can share with the class (valid for 72 hours).

In [ ]:
!python app/demo.py --share